# Three-group two-stage comparison on common samples

Retrain AEF ET-KNN, AEF RF-RF, and conventional optical-topographic RF-RF using the same 119,736 common samples. These results are separate from the original full-sample AEF experiment.

Fill in `COMMON_PATH` and `OUTPUT_PARENT` in the configuration cell before running. No local paths are supplied. Open this notebook in VS Code or Jupyter, select the Python environment, and run all cells. Dependencies: NumPy, pandas, scikit-learn, matplotlib, joblib, and threadpoolctl. tqdm is optional; a built-in progress display is used otherwise. The recorded experiment used scikit-learn 1.5.2; StratifiedGroupKFold requires version 1.1 or later.

The approximately 75% training / 25% testing partition is stratified by label year and target class, grouping locations rounded to six decimal places across years. The first fold of four is a fixed holdout, not four-fold cross-validation. Sample assignments and their digest are saved and shared across all groups.

ET and RF use 300 trees. KNN uses seven neighbors with distance weighting. Its scaler is fitted only on forest training samples. No model selection or hyperparameter tuning uses the test set.

Metrics include OA, Kappa, Macro-F1, Weighted-F1, and binary forest F1. All stages report confusion matrices (rows: reference; columns: prediction), per-class precision, recall, F1, and support. Stage 2 is independently evaluated on reference-labelled forest test samples; final six-class inference routes samples using stage-1 predictions and preserves error propagation.

Efficiency is supplementary: training is measured once; full-test-set prediction is warmed up and repeated three times, using the median. Scaling and batched inference are included. GEE extraction, downloads, and AEF pretraining are excluded. Compressed model size in MiB does not measure peak memory.

The progress display advances after each training stage. Reruns create new timestamped result directories. Each group is saved when complete; original data are unchanged. Outputs include comparison tables, timing records, sample assignments, per-sample predictions, confusion matrices, model files, and parameter records.


In [ ]:
from pathlib import Path
from datetime import datetime
import os,sys,time,json,hashlib,platform,gc
try:
    import numpy as np
    import pandas as pd
    import sklearn
    import matplotlib.pyplot as plt
    import joblib
    from sklearn.ensemble import RandomForestClassifier,ExtraTreesClassifier
    from sklearn.neighbors import KNeighborsClassifier
    from sklearn.preprocessing import StandardScaler
    from sklearn.model_selection import StratifiedGroupKFold
    from sklearn.metrics import accuracy_score,cohen_kappa_score,f1_score,confusion_matrix,classification_report
    from threadpoolctl import threadpool_limits,threadpool_info
except ImportError as exc:
    raise RuntimeError('Install dependencies in a separate cell: %pip install numpy pandas scikit-learn matplotlib joblib threadpoolctl. Restart the kernel before running all cells. StratifiedGroupKFold requires scikit-learn >= 1.1.') from exc

COMMON_PATH = ""
OUTPUT_PARENT = ""
SEED=42
N_TREES=300
K_NEIGHBORS=7
N_JOBS=max(1,min(4,(os.cpu_count() or 2)-1))
PREDICT_BATCH_SIZE=2048
TIMING_REPEATS=3
CLASS5=['EBF','DBF','ENF','DNF','MF']
CLASS6=CLASS5+['Others']
META=['source_id','source_dataset','label_year','feature_year','lc_code','label_binary','final_class','lon','lat']
AEF_FEATURES=['A{:02d}'.format(i) for i in range(64)]
BANDS=['B2','B3','B4','B5','B6','B7','B8','B8A','B11','B12']
TRAD_FEATURES=['Q{}_{}'.format(q,b) for q in range(1,5) for b in BANDS+['NDVI','EVI','NDMI','NBR','NDRE']]+['elevation','slope_deg','aspect_sin','aspect_cos']
FOREST_MAP={50:'EBF',60:'DBF',70:'ENF',80:'DNF',90:'MF'}
GROUPS=[('AEF_main_ET_KNN','AEF', 'ET','KNN'),('AEF_control_RF_RF','AEF','RF','RF'),('Traditional_RF_RF','Traditional','RF','RF')]
STAGES=['step1_binary','step2_true_forest','final_six_class']


def require_path(value, setting):
    if value is None or not str(value).strip():
        raise ValueError("Set " + setting + " before running this notebook.")
    return Path(value)


def sha256_file(path):
    h=hashlib.sha256()
    with path.open('rb') as f:
        for b in iter(lambda:f.read(1024*1024),b''):h.update(b)
    return h.hexdigest()


def export_csv(df,path):
    df.to_csv(path,index=False,encoding='utf-8-sig')


def evaluate(y_true,y_pred,labels):
    return {'OA':accuracy_score(y_true,y_pred),
            'Kappa':cohen_kappa_score(y_true,y_pred),
            'Macro_F1':f1_score(y_true,y_pred,labels=labels,average='macro',zero_division=0),
            'Weighted_F1':f1_score(y_true,y_pred,labels=labels,average='weighted',zero_division=0)}


def save_evaluation(y_true,y_pred,labels,names,stage,out,group):
    cm=confusion_matrix(y_true,y_pred,labels=labels)
    cm_df=pd.DataFrame(cm,index=names,columns=names)
    cm_df.index.name='True_class'
    cm_df.to_csv(out/(stage+'_confusion_counts.csv'),encoding='utf-8-sig')
    norm=np.divide(cm,cm.sum(axis=1,keepdims=True),out=np.zeros(cm.shape,dtype=float),where=cm.sum(axis=1,keepdims=True)!=0)
    norm_df=pd.DataFrame(norm,index=names,columns=names);norm_df.index.name='True_class'
    norm_df.to_csv(out/(stage+'_confusion_row_normalized.csv'),encoding='utf-8-sig')
    report=classification_report(y_true,y_pred,labels=labels,target_names=names,output_dict=True,zero_division=0)
    export_csv(pd.DataFrame([{'class':name,**report[name]} for name in names]),out/(stage+'_per_class_metrics.csv'))
    fig,axes=plt.subplots(1,2,figsize=(12,5),constrained_layout=True)
    for ax,values,title in zip(axes,[cm,norm],['Counts','Recall by true class']):
        im=ax.imshow(values,cmap='Blues',vmin=0,vmax=None if title=='Counts' else 1)
        ax.set_xticks(range(len(names)));ax.set_yticks(range(len(names)))
        ax.set_xticklabels(names,rotation=45,ha='right');ax.set_yticklabels(names)
        ax.set_xlabel('Predicted class');ax.set_ylabel('True class');ax.set_title(title)
        for i in range(len(names)):
            for j in range(len(names)):
                text=str(int(values[i,j])) if title=='Counts' else '{:.1%}'.format(values[i,j])
                threshold=(values.max()/2) if title=='Counts' else .5
                ax.text(j,i,text,ha='center',va='center',color='white' if values[i,j]>threshold else 'black',fontsize=9)
        fig.colorbar(im,ax=ax,fraction=.046,pad=.04)
    fig.suptitle(group+' / '+stage)
    fig.savefig(out/(stage+'_confusion.png'),dpi=300,bbox_inches='tight');plt.close(fig)
    return evaluate(y_true,y_pred,labels)


# Use tqdm when installed; otherwise show a built-in text progress bar.
try:
    from tqdm import tqdm
except ImportError:
    import time
    class tqdm:
        def __init__(self,iterable=None,total=None,desc='',unit='',unit_scale=False,leave=True,**kwargs):
            self.iterable=iterable;self.total=total if total is not None else len(iterable)
            self.desc=desc;self.unit=unit;self.n=0;self.last=0;self.suffix=''
        def __enter__(self):
            self.show(True);return self
        def __exit__(self,*args):
            self.show(True);print('',flush=True)
        def __iter__(self):
            self.show(True)
            for item in self.iterable:
                yield item
                self.update(1)
            self.show(True);print('',flush=True)
        def update(self,n=1):
            self.n+=n;self.show()
        def set_postfix_str(self,text):
            self.suffix=text;self.show(True)
        def show(self,force=False):
            now=time.monotonic()
            if not force and now-self.last<0.3:return
            ratio=min(1,self.n/max(self.total,1));filled=int(ratio*20)
            print('\r{} [{}{}] {:6.1%} {}/{} {} {}'.format(
                self.desc,'█'*filled,'·'*(20-filled),ratio,self.n,self.total,self.unit,self.suffix),end='',flush=True)
            self.last=now



In [ ]:

def predict_batched(model,scaler,X,progress=False,desc='Prediction'):
    if len(X)==0:return np.array([],dtype=object)
    ranges=range(0,len(X),PREDICT_BATCH_SIZE)
    if progress:ranges=tqdm(ranges,desc=desc,unit='batch',leave=False)
    chunks=[]
    for start in ranges:
        part=X[start:start+PREDICT_BATCH_SIZE]
        if scaler is not None:part=scaler.transform(part)
        chunks.append(model.predict(part))
    return np.concatenate(chunks)


def cascade_predict(model1,model2,scaler2,X):
    binary=predict_batched(model1,None,X)
    pred=np.full(len(X),'Others',dtype=object)
    routed=binary==1
    if routed.any():pred[routed]=predict_batched(model2,scaler2,X[routed])
    return binary,pred


def time_prediction(action,repeats):
    # One full untimed warm-up, followed by repeated full-test-set measurements.
    reference=action();times=[]
    for _ in range(repeats):
        t=time.perf_counter();result=action();times.append(time.perf_counter()-t)
        first=reference if isinstance(reference,tuple) else (reference,)
        last=result if isinstance(result,tuple) else (result,)
        if not all(np.array_equal(a,b) for a,b in zip(first,last)):
            raise RuntimeError('Repeated predictions are inconsistent. Stopping timing aggregation.')
    return reference,float(np.median(times)),times


def build_tree(name,n_trees):
    cls=ExtraTreesClassifier if name=='ET' else RandomForestClassifier
    return cls(n_estimators=n_trees,criterion='gini',max_depth=None,max_features='sqrt',
        min_samples_split=2,min_samples_leaf=1,bootstrap=(name=='RF'),class_weight=None,
        random_state=SEED,n_jobs=N_JOBS)


def validate_data(df,expected_count):
    missing=set(META+AEF_FEATURES+TRAD_FEATURES)-set(df.columns)
    if missing:raise ValueError('Missing fields in the common sample table: '+str(sorted(missing)))
    if df.columns.duplicated().any():raise ValueError('Duplicate column names.')
    if expected_count is not None and len(df)!=expected_count:raise ValueError('Expected {} common samples; found {}.'.format(expected_count,len(df)))
    if df.source_id.isna().any() or df.source_id.duplicated().any():raise ValueError('source_id contains missing or duplicate values.')
    if df.final_class.isna().any() or set(df.final_class)!=set(CLASS6):raise ValueError('The six-class labels are incomplete.')
    for c in AEF_FEATURES+TRAD_FEATURES+['lon','lat','lc_code','label_binary','label_year','feature_year']:
        df[c]=pd.to_numeric(df[c],errors='raise')
    for c in ['lc_code','label_binary','label_year','feature_year']:
        vals=df[c].to_numpy(float)
        if not np.isfinite(vals).all() or not (vals==np.floor(vals)).all():raise ValueError('Invalid integer label field: '+c)
        df[c]=df[c].astype(int)
    if not np.isfinite(df[AEF_FEATURES+TRAD_FEATURES].to_numpy(float)).all() or (df[AEF_FEATURES+TRAD_FEATURES]==-9999).any().any():raise ValueError('Common features contain missing or invalid values. Stopping without dropping rows.')
    if not np.isfinite(df[['lon','lat']].to_numpy(float)).all() or df.lon.abs().gt(180).any() or df.lat.abs().gt(90).any():raise ValueError('Invalid longitude or latitude values.')
    if not df.lc_code.map(FOREST_MAP).fillna('Others').eq(df.final_class).all() or not df.label_binary.eq(df.final_class.ne('Others').astype(int)).all():raise ValueError('Classification labels are inconsistent.')
    if not (((df.label_year==2015)&(df.feature_year==2017))|((df.label_year==2020)&(df.feature_year==2020))).all():raise ValueError('Invalid reference-label and feature-year combination.')
    expected_source=np.where(df.label_year==2015,'GLC2015','SRS2020')
    if not (df.source_dataset.to_numpy()==expected_source).all():raise ValueError('Sample sources do not match their reference-label years.')
    return df.sort_values('source_id',kind='mergesort').reset_index(drop=True)


def shared_split(df,out):
    group=np.array([format(x,'.6f')+'_'+format(y,'.6f') for x,y in zip(df.lon,df.lat)])
    strata=df.label_year.astype(str)+'_'+df.final_class
    if strata.value_counts().min()<4:raise ValueError('A year-class stratum has fewer than four records; stratification cannot use the current settings.')
    # First fold selected in advance; no model results used to select the split.
    splitter=StratifiedGroupKFold(n_splits=4,shuffle=True,random_state=SEED)
    train,test=next(splitter.split(np.zeros(len(df)),strata,group))
    if set(group[train])&set(group[test]):raise RuntimeError('Coincident-location samples occur in both training and testing.')
    if set(train)&set(test) or len(train)+len(test)!=len(df):raise RuntimeError('The split overlaps or does not cover all samples.')
    for subset in [train,test]:
        if set(df.iloc[subset].final_class)!=set(CLASS6):raise ValueError('A subset is missing one or more target classes.')
    split=df[META].copy();split['location_group']=group;split['split']='train';split.loc[test,'split']='test'
    export_csv(split,out/'shared_sample_split.csv')
    counts=split.groupby(['split','label_year','final_class']).size().rename('n').reset_index()
    export_csv(counts,out/'shared_split_counts.csv')
    export_csv(split.loc[pd.Series(group).duplicated(keep=False)],out/'duplicate_location_split_check.csv')
    digest=hashlib.sha256('\n'.join(split.source_id+'|'+split['split']).encode('utf-8')).hexdigest()
    return train,test,digest


def comparison_panels(out):
    for stage in STAGES:
        labels=['Nonforest','Forest'] if stage=='step1_binary' else CLASS5 if stage=='step2_true_forest' else CLASS6
        fig,axes=plt.subplots(1,3,figsize=(17,5),constrained_layout=True)
        for ax,(name,_,_,_) in zip(axes,GROUPS):
            cm=pd.read_csv(out/name/(stage+'_confusion_row_normalized.csv'),index_col=0).to_numpy(float)
            im=ax.imshow(cm,vmin=0,vmax=1,cmap='Blues')
            ax.set_xticks(range(len(labels)));ax.set_yticks(range(len(labels)))
            ax.set_xticklabels(labels,rotation=45,ha='right');ax.set_yticklabels(labels)
            ax.set_title(name);ax.set_xlabel('Predicted class');ax.set_ylabel('True class')
            for i in range(len(labels)):
                for j in range(len(labels)):
                    ax.text(j,i,'{:.1%}'.format(cm[i,j]),ha='center',va='center',color='white' if cm[i,j]>.5 else 'black',fontsize=8)
        fig.colorbar(im,ax=axes.ravel().tolist(),fraction=.02,pad=.02)
        fig.suptitle(stage+' / shared test samples / row-normalized')
        fig.savefig(out/('three_groups_'+stage+'_confusion.png'),dpi=300,bbox_inches='tight');plt.close(fig)


def run_comparison(input_path=None,output_parent=None,expected_count=119736,n_trees=None,timing_repeats=None):
    input_path = require_path(input_path if input_path is not None else COMMON_PATH, 'COMMON_PATH')
    output_parent = require_path(output_parent if output_parent is not None else OUTPUT_PARENT, 'OUTPUT_PARENT')
    if not input_path.is_file():raise FileNotFoundError('Common sample table not found: '+str(input_path))
    out=output_parent/('comparison_'+datetime.now().strftime('%Y%m%d_%H%M%S_%f'))
    out.mkdir(parents=True,exist_ok=False)
    trees=n_trees or N_TREES;repeats=timing_repeats or TIMING_REPEATS
    notes=[]
    def log(s):
        print(s,flush=True);notes.append(s);(out/'summary.txt').write_text('\n'.join(notes),encoding='utf-8')
    log('Input: '+str(input_path));log('Result directory: '+str(out))
    log('All three groups are retrained using one shared split. The progress bar updates after each training stage finishes.')
    all_metrics=[];runtimes=[]
    with tqdm(total=15,desc='Overall progress',unit='stage') as bar, sklearn.config_context(working_memory=256), threadpool_limits(limits=N_JOBS):
        t=time.perf_counter();df=pd.read_csv(input_path,encoding='utf-8-sig',dtype={'source_id':str})
        df=validate_data(df,expected_count);load_seconds=time.perf_counter()-t
        bar.update(1);bar.set_postfix_str('Shared sample split')
        train,test,split_hash=shared_split(df,out)
        binary=df.label_binary.to_numpy(int);classes=df.final_class.to_numpy()
        tf=train[binary[train]==1];vf=test[binary[test]==1]
        log('Common samples {}; training {}; testing {}; forest training {}; reference-labelled forest testing {}.'.format(len(df),len(train),len(test),len(tf),len(vf)))
        bar.update(1)
        runconfig={'input':str(input_path),'input_sha256':sha256_file(input_path),'seed':SEED,'n_samples':len(df),
            'n_train':len(train),'n_test':len(test),'n_train_forest':len(tf),'n_test_forest':len(vf),
            'split_sha256':split_hash,'split_method':'StratifiedGroupKFold,4fold,shuffle=True,seed42,first fold; strata=label_year+final_class; location rounded6dp across years.',
            'input_features':{'AEF':AEF_FEATURES,'Traditional':TRAD_FEATURES},'n_estimators':trees,'k_neighbors':K_NEIGHBORS,
            'n_jobs':N_JOBS,'prediction_batch_size':PREDICT_BATCH_SIZE,'timing_repeats':repeats,
            'python':sys.version,'executable':sys.executable,'sklearn':sklearn.__version__,'numpy':np.__version__,'pandas':pd.__version__,
            'platform':platform.platform(),'cpu_count':os.cpu_count(),'threadpools':threadpool_info(),'load_validate_seconds':load_seconds,
            'timing_policy':'Sequential groups; one measured fit per stage including scaler fitting where applicable; full-dataset untimed prediction warmup then repeated median wall-clock times with identical batch size. Includes scaler.transform at inference. No GEE extraction or AEF pretraining cost. Model size uses joblib compression3 in MiB; not peak RAM.',
            'validation_scope':'Coincident-location group holdout; not spatial-block or independent external validation. No hyperparameter search or model selection on this test set.',
            'original_results':'Separate historical full-sample experiment, not copied into these comparison tables.'}
        (out/'run_config.json').write_text(json.dumps(runconfig,ensure_ascii=False,indent=2),encoding='utf-8')
        for group,kind,first,second in GROUPS:
            gd=out/group;gd.mkdir()
            features=AEF_FEATURES if kind=='AEF' else TRAD_FEATURES
            X=df[features].to_numpy(dtype=np.float64)
            Xtrain=np.ascontiguousarray(X[train]);Xtest=np.ascontiguousarray(X[test]);Xtf=np.ascontiguousarray(X[tf]);Xvf=np.ascontiguousarray(X[vf])
            log(group+': training stage 1 '+first+'……');bar.set_postfix_str(group+' stage 1 training')
            model1=build_tree(first,trees)
            t=time.perf_counter();model1.fit(Xtrain,binary[train]);fit1=time.perf_counter()-t;bar.update(1)
            log(group+': training stage 2 '+second+'……');bar.set_postfix_str(group+' stage 2 training')
            scaler=None
            t=time.perf_counter()
            if second=='KNN':
                scaler=StandardScaler();Z=scaler.fit_transform(Xtf)
                model2=KNeighborsClassifier(n_neighbors=K_NEIGHBORS,weights='distance',algorithm='auto',metric='minkowski',p=2,n_jobs=N_JOBS)
                model2.fit(Z,classes[tf]);del Z
            else:
                model2=build_tree(second,trees);model2.fit(Xtf,classes[tf])
            fit2=time.perf_counter()-t;bar.update(1)
            log(group+': prediction and efficiency measurement ({} repeats after warm-up)...'.format(repeats));bar.set_postfix_str(group+' prediction and evaluation')
            # Batch progress for predictions; benchmarks below do not print inside timed intervals.
            p1=predict_batched(model1,None,Xtest,True,group+' stage 1')
            p2=predict_batched(model2,scaler,Xvf,True,group+' stage 2')
            with tqdm(total=3,desc=group+' timing tasks',unit='task',leave=False) as tb:
                p1_ref,sec1,raw1=time_prediction(lambda:predict_batched(model1,None,Xtest),repeats);tb.update(1)
                p2_ref,sec2,raw2=time_prediction(lambda:predict_batched(model2,scaler,Xvf),repeats);tb.update(1)
                cascade,secfinal,rawfinal=time_prediction(lambda:cascade_predict(model1,model2,scaler,Xtest),repeats);tb.update(1)
            cp1,final=cascade
            if not np.array_equal(p1,p1_ref) or not np.array_equal(p2,p2_ref) or not np.array_equal(cp1,p1):raise RuntimeError('Independent and cascade predictions are inconsistent.')
            for stage,yt,yp,labels,names in [
                ('step1_binary',binary[test],p1,[0,1],['Nonforest','Forest']),
                ('step2_true_forest',classes[vf],p2,CLASS5,CLASS5),
                ('final_six_class',classes[test],final,CLASS6,CLASS6)]:
                metrics=save_evaluation(yt,yp,labels,names,stage,gd,group)
                metrics.update(Group=group,Stage=stage,N_test=len(yt),Feature_set=kind,Step1_model=first,Step2_model=second,Split_SHA256=split_hash)
                metrics['Forest_F1']=f1_score(yt,yp,pos_label=1,zero_division=0) if stage=='step1_binary' else np.nan
                all_metrics.append(metrics)
            export_csv(pd.DataFrame(all_metrics),out/'all_groups_all_stages_metrics.csv')
            pred=df.iloc[test][META].reset_index(drop=True).copy();pred['step1_pred_binary']=p1
            pred['step2_pred_on_true_forest']='';pred.loc[binary[test]==1,'step2_pred_on_true_forest']=p2
            pred['cascade_pred_class']=final;export_csv(pred,gd/'test_predictions.csv')
            runtime={'Group':group,'Step1_fit_seconds':fit1,'Step2_fit_seconds':fit2,'Total_fit_seconds':fit1+fit2,
                'Step1_predict_median_seconds':sec1,'Step2_true_forest_predict_median_seconds':sec2,
                'Cascade_predict_median_seconds':secfinal,'Cascade_samples_per_second':len(test)/max(secfinal,1e-12),
                'N_test':len(test),'N_test_true_forest':len(vf),'N_predicted_forest':int((p1==1).sum()),'Timing_repeats':repeats,
                'N_jobs':N_JOBS,'Prediction_batch_size':PREDICT_BATCH_SIZE,'Split_SHA256':split_hash}
            (gd/'timing_repetitions.json').write_text(json.dumps({'step1_predict_seconds':raw1,'step2_true_forest_predict_seconds':raw2,'cascade_predict_seconds':rawfinal},indent=2),encoding='utf-8')
            bar.update(1);bar.set_postfix_str(group+' saving models')
            joblib.dump({'group':group,'step1':model1,'step2':model2,'step2_scaler':scaler,'features':features,'class5':CLASS5,'split_sha256':split_hash},gd/'models.joblib',compress=3)
            runtime['Saved_models_MiB']=(gd/'models.joblib').stat().st_size/(1024**2)
            runtimes.append(runtime);export_csv(pd.DataFrame(runtimes),out/'efficiency_comparison.csv')
            (gd/'model_parameters.json').write_text(json.dumps({'step1':model1.get_params(),'step2':model2.get_params(),'step2_scaler':'StandardScaler fitted ONLY on forest training samples' if scaler else None},ensure_ascii=False,indent=2),encoding='utf-8')
            log('{} completed: final OA={:.4f}; training {:.2f} s; median cascade prediction {:.2f} s.'.format(group,all_metrics[-1]['OA'],fit1+fit2,secfinal))
            del model1,model2,scaler,X,Xtrain,Xtest,Xtf,Xvf;gc.collect();bar.update(1)
        bar.set_postfix_str('Preparing three-group tables and figures')
        combined=pd.DataFrame(all_metrics)
        order=['Group','Stage','Feature_set','Step1_model','Step2_model','N_test','OA','Kappa','Macro_F1','Weighted_F1','Forest_F1','Split_SHA256']
        combined=combined[order];export_csv(combined,out/'all_groups_all_stages_metrics.csv')
        for stage in STAGES:
            export_csv(combined.loc[combined.Stage==stage],out/('comparison_'+stage+'.csv'))
        comparison_panels(out)
        text=['Three-group two-stage comparison on common samples. All proportions are stored on a 0-1 scale.',
            'Stage 1 evaluates forest/non-forest discrimination on all test samples; stage 2 evaluates five forest types on reference-labelled forest test samples.',
            'Final six-class prediction routes only samples predicted as forest at stage 1 to stage 2, preserving error propagation. Reference forest labels must not be used as an inference mask.',
            'Per-class precision is UA and recall is PA; F1 and support are also reported. Confusion-matrix rows are reference classes and columns are predicted classes.',
            'The approximately 75%/25% split is stratified by label year and target class. Locations rounded to six decimal places are grouped across years. This is not spatial-block validation.',
            'Only the prespecified first fold is used as a holdout test set; results are not four-fold cross-validation averages. This test set is not used for model selection or hyperparameter tuning.',
            'ET and RF use 300 trees; KNN uses seven neighbors with distance weighting. See each group model_parameters.json for all parameters.',
            'The AEF main configuration fits the stage-2 scaler on forest training samples only. RF and ET are not standardized. AEF and conventional features are not combined.',
            'Efficiency: one measured fit; prediction median from three full-test-set repeats after warm-up, including KNN scaling and batched inference. GEE extraction, downloads, and AEF pretraining are excluded.',
            'Serialized model size uses MiB (1024^2 bytes) with joblib compression level 3. It does not measure peak memory usage.',
            'Original full-sample AEF results remain separate. All results from this run are retrained on common samples.',
            'Reruns create new timestamped directories. The partition is fixed for the same data, software versions, and settings. Original files are not modified.']
        (out/'README.txt').write_text('\n'.join(text),encoding='utf-8')
        log('All three groups completed. Shared split digest: '+split_hash)
        log('Accuracy summary:\n'+combined[['Group','Stage','N_test','OA','Kappa','Macro_F1','Weighted_F1','Forest_F1']].to_string(index=False))
        log('Outputs include comparison_*.csv, efficiency_comparison.csv, shared_split_counts.csv, and three_groups_*_confusion.png.')
        bar.update(1);bar.set_postfix_str('Complete')
    return out,combined,pd.DataFrame(runtimes)


In [ ]:
RESULT_DIR, ALL_METRICS, EFFICIENCY_TABLE = run_comparison()
